# 2b–2c — Variational autoencoder on hand-drawn fish

Dataset: 2,000 black-and-white 28x28 fish from Quick, Draw! (`data/fish.npz`, see `docs/vae_decisions.md`).

The VAE is in `src/tp5/vae.py`. It reuses the layers, activations and optimizers of `Autoencoder`, but the encoder outputs `mu` and `logvar` (two heads on the same hidden layer), the latent code is sampled as `z = mu + exp(0.5*logvar) * eps`, and the loss is the cross-entropy summed over pixels plus the KL divergence to `N(0, I)`.

In [ ]:
import sys
from pathlib import Path

# make `tp5` importable without installing the package: add the repo's src/ folder
for root in (Path.cwd(), *Path.cwd().parents):
    if (root / "src" / "tp5").is_dir():
        sys.path.insert(0, str(root / "src"))
        break

import matplotlib.pyplot as plt
import numpy as np

from tp5.fish import load_fish, show_images
from tp5.vae import VAE

X = load_fish().astype(float)
X.shape, X.mean()

## 1. Gradient check

Compare the analytic gradients with finite differences on a small network, with the noise `eps` fixed so both sides use the same sample. The relative error should be far below 1e-5.

In [ ]:
rng = np.random.default_rng(0)
Xs = (rng.random((8, 20)) < 0.3).astype(float)
small = VAE(20, (12, 6), 2, seed=1, activation="tanh")


def total_loss():
    small.rng = np.random.default_rng(5)  # same eps on every call
    mu, lv = small.encode(Xs)
    _, _, z = small.sampling((mu, lv), small.rng)
    X_hat = small._forward(z, start=small.latent_index + 1)[-1]
    return sum(small.loss_parts(Xs, X_hat))


small.rng = np.random.default_rng(5)
grads = small.gradients(Xs)
worst = 0.0
for p, g in zip(small.params, grads):
    for _ in range(5):
        idx = tuple(rng.integers(0, s) for s in p.shape)
        old, h = p[idx], 1e-5
        p[idx] = old + h
        a = total_loss()
        p[idx] = old - h
        b = total_loss()
        p[idx] = old
        num = (a - b) / (2 * h)
        worst = max(worst, abs(num - g[idx]) / (abs(num) + abs(g[idx]) + 1e-8))
print(f"worst relative error: {worst:.2e}")
assert worst < 1e-5

## 2. Training

Architecture: 784 → 256 → 64 → (mu, logvar) of size 2 → 64 → 256 → 784, tanh in the hidden layers, sigmoid output, Adam.

In [ ]:
vae = VAE(784, (256, 64), 2, seed=0, activation="tanh")
hist = vae.fit(X, epochs=200, batch_size=64, log_every=5, seed=0)
print(f"recon {vae.recon_log[-1]:.1f}   KL {vae.kl_log[-1]:.1f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(hist["epoch"], vae.recon_log)
ax1.set(title="Reconstruction (BCE summed over pixels)", xlabel="epoch")
ax2.plot(hist["epoch"], vae.kl_log)
ax2.set(title="KL to N(0, I)", xlabel="epoch")
fig.tight_layout()

Healthy training: the reconstruction term falls and the KL term rises above 0 and then levels off. If KL stayed near 0 the posterior would have collapsed (every image mapped to the same distribution).

## 3. Reconstructions

Top: originals. Bottom: reconstruction from `mu` (no noise).

In [ ]:
n = 10
show_images(np.vstack([X[:n], vae.reconstruct(X[:n])]), columns=n);

## 4. The latent space

Each point is the mean `mu` of one fish. The prior pulls the cloud toward a blob around (0, 0).

In [ ]:
mu, logvar = vae.encode(X)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(mu[:, 0], mu[:, 1], s=4, alpha=0.5)
ax.set(xlabel="z1", ylabel="z2", title="Encoded fish (mu)")
ax.set_aspect("equal")
print("mean of mu:", mu.mean(0).round(2), " std of mu:", mu.std(0).round(2))
print("mean sigma:", np.exp(0.5 * logvar).mean(0).round(2))

Decode a regular grid of `z` values. Neighbouring cells should change smoothly, which is the point of the variational scheme.

In [ ]:
k = 10
axis = np.linspace(-2.5, 2.5, k)
grid = np.array([[a, b] for b in axis[::-1] for a in axis])  # z1 left to right, z2 top to bottom
show_images(vae.decode(grid), columns=k);

## 5. Generating new fish (2c)

Draw `z` from the prior `N(0, I)` and decode it. Below each sample we show the closest training image (fewest differing pixels) to check that the generated fish is new and not a copy.

In [ ]:
gen_rng = np.random.default_rng(1)
z = gen_rng.standard_normal((10, 2))
samples = vae.decode(z)

dist = (np.round(samples)[:, None, :] != X[None, :, :]).sum(axis=2)  # (10, 2000) differing pixels
nearest = dist.argmin(axis=1)
print("pixels differing from nearest training image:", dist.min(axis=1))
show_images(
    np.vstack([samples, X[nearest]]),
    titles=["generated"] * 10 + ["nearest in data"] * 10,
    columns=10,
);

## 6. Hyperparameter sweep

So far we used one hand-picked configuration. Here we search for a better one, **one factor at a time** (a full grid would take hours): each stage fixes the winner of the previous stage.

| Stage | What changes |
|---|---|
| 1 | latent size: 2, 8, 16 |
| 2 | learning rate and number of epochs |
| 3 | KL weight `beta` |
| 4 | network width and hidden activation |

**Protocol.** 200 of the 2,000 fish are held out (`split`). Every configuration is trained on the other 1,800 fish with 2 seeds. We compare the **validation** `recon + KL` (the beta = 1 loss, so runs with different `beta` stay comparable), the two parts separately, and the number of active latent units (variance of `mu` above 0.01). A lower number is better, but it does not measure how fish-like the samples are, so every stage also shows 10 samples drawn from N(0, I).

Results are saved in `results/sweep_*.json`, so re-running the notebook only loads them. Set `RERUN = True` to train again (stages take 5 to 15 minutes), or `QUICK = True` for a 2-epoch smoke test.

In [ ]:
import json
from pathlib import Path

from tp5.vae_study import run_config, split, summarize

QUICK = False  # True: 2 epochs and one seed, only to check that the notebook runs
RERUN = False  # True: ignore saved results and train again

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "tp5").is_dir())
RESULTS = ROOT / "results"
X_train, X_val = split(X)
print(len(X_train), "training fish,", len(X_val), "validation fish")


def sweep(name, configs, **fixed):
    """Train every config in `configs` ({label: run_config arguments}) and return the rows.

    `fixed` holds the arguments shared by all configs (the winners of earlier
    stages). Rows are cached in results/sweep_<name>.json, together with 10
    samples from the first seed's model (decoded from the same random z).
    """
    path = RESULTS / f"sweep_{name}{'_quick' if QUICK else ''}.json"
    if path.exists() and not RERUN:
        return json.loads(path.read_text())
    rows = []
    for label, cfg in configs.items():
        args = {"epochs": 50, "seeds": (0, 1), **fixed, **cfg}
        if QUICK:
            args["epochs"], args["seeds"] = 2, (0,)
        for r in run_config(X_train, X_val, **args):
            model = r.pop("model")
            r["label"], r["hidden"] = label, list(r["hidden"])
            z = np.random.default_rng(1).standard_normal((10, r["latent_dim"]))
            r["samples"] = np.round(model.decode(z), 2).tolist() if r["seed"] == args["seeds"][0] else None
            rows.append(r)
            print(f"{label}, seed {r['seed']}: val recon {r['val_recon']:.1f}, KL {r['val_kl']:.1f}, "
                  f"total {r['val_total']:.1f}, {r['seconds']:.0f}s", flush=True)
    RESULTS.mkdir(exist_ok=True)
    path.write_text(json.dumps(rows))
    return rows


def report(rows):
    """Print the summary table, plot validation total with error bars, show samples."""
    summary = summarize(rows, "label")
    print(f"{'config':<26}{'val total':>11}{'+-':>6}{'recon':>8}{'KL':>7}{'active':>8}")
    for s in summary:
        print(f"{s['label']:<26}{s['val_total']:>11.1f}{s['val_total_std']:>6.1f}"
              f"{s['val_recon']:>8.1f}{s['val_kl']:>7.1f}{s['active_units']:>8.1f}")
    fig, ax = plt.subplots(figsize=(6, 0.5 + 0.45 * len(summary)))
    ax.barh([s["label"] for s in summary], [s["val_total"] for s in summary],
            xerr=[s["val_total_std"] for s in summary], color="#1B6CA8")
    ax.invert_yaxis()
    ax.set(xlabel="validation recon + KL (lower is better)")
    lo = min(s["val_total"] for s in summary)
    ax.set_xlim(lo - 25, max(s["val_total"] for s in summary) + 10)
    fig.tight_layout()
    shown = [r for r in rows if r["samples"] is not None]
    fig, axes = plt.subplots(len(shown), 10, figsize=(10, 1.15 * len(shown)), squeeze=False)
    for row_axes, r in zip(axes, shown):
        for a, img in zip(row_axes, r["samples"]):
            a.imshow(np.array(img).reshape(28, 28), cmap="binary", vmin=0, vmax=1)
            a.axis("off")
        row_axes[0].set_title(r["label"], fontsize=8, loc="left")
    fig.tight_layout()


def best(rows, configs):
    """Return the run_config arguments of the config with the lowest mean validation total."""
    summary = summarize(rows, "label")
    return configs[min(summary, key=lambda s: s["val_total"])["label"]]

### Stage 1: latent size

The 2D model is the one we plot, but a larger latent space can describe eye position, fin shape and direction at the same time.

In [ ]:
LATENT = {f"latent {d}": {"latent_dim": d} for d in (2, 8, 16)}
rows = sweep("latent", LATENT)
report(rows)
base = best(rows, LATENT)
print("best so far:", base)

### Stage 2: learning rate and epochs

At the best latent size. Longer training and a smaller learning rate usually go together, so we try all four combinations.

In [ ]:
TRAIN = {f"lr {lr:g}, {ep} epochs": {"lr": lr, "epochs": ep} for lr in (1e-3, 3e-4) for ep in (50, 150)}
rows = sweep("training", TRAIN, **base)
report(rows)
base = {**base, **best(rows, TRAIN)}
print("best so far:", base)

### Stage 3: KL weight `beta`

`beta` multiplies the KL term in the training loss. Below 1 the model is allowed to keep more information about each image (sharper reconstructions) at the price of a latent space that matches N(0, I) less well, which can hurt the samples. The validation total is always the plain `recon + KL`, so it favours `beta = 1` by construction: for this stage look at the samples and at the recon and KL columns as well.

In [ ]:
BETA = {f"beta {b:g}": {"beta": b} for b in (0.5, 1.0)}
rows = sweep("beta", BETA, **base)
report(rows)
base = {**base, **best(rows, BETA)}
print("best so far:", base)

### Stage 4: width and activation

In [ ]:
ARCH = {
    "256-64 tanh": {"hidden": (256, 64), "activation": "tanh"},
    "512-128 tanh": {"hidden": (512, 128), "activation": "tanh"},
    "256-64 relu": {"hidden": (256, 64), "activation": "relu"},
}
rows = sweep("architecture", ARCH, **base)
report(rows)
base = {**base, **best(rows, ARCH)}
print("final configuration:", base)

### Final model

Retrain the winning configuration (with the number of epochs chosen in stage 2) on the training fish and look at it on the held-out fish. In our run, training for 200 epochs instead of 150 scored worse on the held-out fish (184.0 vs 178.1, one seed): the model starts to overfit, so we keep the 150 epochs from the sweep.

In [ ]:
RUN_FINAL = True  # training takes a few minutes

if RUN_FINAL:
    final = run_config(X_train, X_val, **{**base, "epochs": 2 if QUICK else base["epochs"], "seeds": (0,)})[0]
    best_vae = final["model"]
    print(f"validation recon {final['val_recon']:.1f}, KL {final['val_kl']:.1f}, total {final['val_total']:.1f}")
    n = 10
    show_images(np.vstack([X_val[:n], best_vae.reconstruct(X_val[:n])]), columns=n);

In [ ]:
if RUN_FINAL:
    z = np.random.default_rng(1).standard_normal((20, best_vae.W_lv.shape[1]))  # W_lv is (hidden, latent)
    show_images(best_vae.decode(z), columns=10);